In [1]:
import json
from pathlib import Path
import polars as pl
from IPython.display import Markdown

In [2]:
from pan26_genai_detection.evaluator.evaluator import c_at_1, auc, f1, f05u, brier_score

def evaluate_all(y_true: pl.DataFrame, y_pred: pl.DataFrame):
    y_true = y_true.to_numpy()
    y_pred = y_pred.to_numpy()
    return pl.from_dict({
        'roc-auc': auc(y_true, y_pred),
        'brier': brier_score(y_true, y_pred),
        'c@1': c_at_1(y_true, y_pred),
        'f1': f1(y_true, y_pred),
        'f05u': f05u(y_true, y_pred),
    }).with_columns(
        pl.mean_horizontal(pl.all()).alias('mean'),
        pl.lit((y_pred[y_true == 0] >= 0.5).sum() / (y_true == 0).sum()).alias('fpr'),
        pl.lit((y_pred[y_true == 1] < 0.5).sum() / (y_true == 1).sum()).alias('fnr'),
    )

In [3]:
def build_truth_map(truth_file: Path, id_map_file: Path = None):
    truth_df = pl.read_ndjson(truth_file)
    if not id_map_file:
        return truth_df.with_columns(pl.lit(None).alias('orig_id'))
    id_map_df = pl.read_ndjson(id_map_file)
    return truth_df.join(id_map_df, on='id', how='left')

def load_run_files(rundir: Path):
    meta_df = pl.read_ndjson(rundir / 'metadata.jsonl')
    runs_df = []
    run_ids = meta_df.select('run_id').unique().to_series()

    # Leave no stray runs behind
    assert len(run_ids) == len(list(rundir.joinpath('outputs-flat').glob('2026-*')))

    for rid in run_ids:
        runfile = rundir / 'outputs-flat' / rid
        for l in runfile.open('r'):
            j = json.loads(l)
            j['run_id'] = rid
            runs_df.append(j)

    df = pl.from_records(runs_df)
    df = df.join(meta_df.select(['run_id', 'team', 'software', 'dataset']), on='run_id', how='left')
    return df

def join_runs_and_truth(runs_df: pl.DataFrame, meta_df: pl.DataFrame):
    df = runs_df.join(meta_df, on='id', how='left')
    df = df.rename({'label': 'label_pred', 'label_right': 'label_truth'})

    # Sanity check
    assert df['label_truth'].is_not_null().all()
    assert df['label_pred'].is_not_null().all()

    # Reorder columns and sort
    col_order = ['team', 'software', 'run_id', 'id', 'orig_id', 'model', 'source', 'label_pred', 'label_truth', 'dataset']
    df = df.select(col_order)
    df = df.sort(by=['team', 'software', 'run_id', 'model', 'label_truth'])
    return df

def load_run_truth_and_meta(base_path, truth_file, id_map_file=None):
    truth_map_pan26 = build_truth_map(truth_file, id_map_file)
    runs_pan26 = load_run_files(base_path)
    return join_runs_and_truth(runs_pan26, truth_map_pan26)

In [4]:
BASE_PATH = Path('../data')
BASE_EVAL_PATH = BASE_PATH / 'task-eval'
BASE_RUN_PATH = BASE_EVAL_PATH / 'tira-runs'

In [5]:
pan25_df = load_run_truth_and_meta(BASE_RUN_PATH / 'pan25',
                                   BASE_PATH / 'splits' / 'pan25-voight-kampff-test-truth.jsonl')
pan25_df

team,software,run_id,id,orig_id,model,source,label_pred,label_truth,dataset
str,str,str,str,null,str,str,f64,i64,str
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""440ff192-f174-59ec-8266-da8bca…",null,"""deepseek-r1-distill-qwen-32b""","""riddell-juola-o1-deepseek""",0.720498,1,"""pan25-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""aeb851b0-c06b-54ba-b645-14b5df…",null,"""deepseek-r1-distill-qwen-32b""","""gutenberg-19c-fiction""",0.719775,1,"""pan25-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""7a9ba40a-e4e0-5375-9f20-fa43e2…",null,"""deepseek-r1-distill-qwen-32b""","""gutenberg-19c-fiction""",0.710772,1,"""pan25-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""79dcf967-b620-5fca-9a1c-4e3b42…",null,"""deepseek-r1-distill-qwen-32b""","""brennan-greenstadt""",0.718746,1,"""pan25-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""85e38931-bb41-5f11-898e-fd7dd2…",null,"""deepseek-r1-distill-qwen-32b""","""brennan-greenstadt""",0.722734,1,"""pan25-generative-ai-detection-…"
…,…,…,…,…,…,…,…,…,…
"""zhengqiaozeng""","""three""","""2026-05-10-22-08-31""","""cd0a01a2-69c8-5c43-8785-dd7748…",null,"""text-bison-002""","""pan24-test""",0.999997,1,"""pan25-generative-ai-detection-…"
"""zhengqiaozeng""","""three""","""2026-05-10-22-08-31""","""e11abbfa-a99c-55a4-850f-0f9957…",null,"""text-bison-002""","""pan24-test""",0.999997,1,"""pan25-generative-ai-detection-…"
"""zhengqiaozeng""","""three""","""2026-05-10-22-08-31""","""25f67501-5396-5d44-9681-921803…",null,"""text-bison-002""","""pan24-test""",0.999997,1,"""pan25-generative-ai-detection-…"


In [6]:
pan26_df = load_run_truth_and_meta(BASE_RUN_PATH / 'pan26',
                                   BASE_PATH / 'splits' / 'pan26-voight-kampff-test-truth.jsonl',
                                   BASE_PATH / 'sampled' / 'pan26-voight-kampff-orig-ids.jsonl')
pan26_df

team,software,run_id,id,orig_id,model,source,label_pred,label_truth,dataset
str,str,str,str,str,str,str,f64,i64,str
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""c64c7947-05e9-5826-aaef-8f9557…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.71872,1,"""pan26-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""a6d5ce11-948f-5935-9319-ee4509…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.719311,1,"""pan26-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""313df7ca-dfb1-5770-8d10-53317d…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.719734,1,"""pan26-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""8972c72b-4069-5792-939f-d87ac2…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.719496,1,"""pan26-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""60cb984e-b516-5d3b-8ff4-6bb8fa…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.718834,1,"""pan26-generative-ai-detection-…"
…,…,…,…,…,…,…,…,…,…
"""zhengqiaozeng""","""three""","""2026-05-08-05-27-50""","""e512c65e-f51a-551c-9f19-b57b12…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…",0.00082,1,"""pan26-generative-ai-detection-…"
"""zhengqiaozeng""","""three""","""2026-05-08-05-27-50""","""b439cb5e-e7e0-5f85-8404-de5408…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…",0.000237,1,"""pan26-generative-ai-detection-…"
"""zhengqiaozeng""","""three""","""2026-05-08-05-27-50""","""b247c664-e1f4-5dd6-a20f-88a553…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…",0.000316,1,"""pan26-generative-ai-detection-…"


In [14]:
def get_results_per_run(df):
    group_cols = ['team', 'software', 'run_id', 'dataset']

    def evaluate_group(key, group_df):
        metrics = evaluate_all(
            group_df.select('label_truth'),
            group_df.select('label_pred'),
        )
        keys = pl.DataFrame([dict(zip(group_cols, key))])
        return pl.concat([keys, metrics], how='horizontal')

    return pl.concat(
        evaluate_group(key, group_df)
        for key, group_df in df.partition_by(
            group_cols,
            as_dict=True,
            maintain_order=True,
        ).items()
    ).sort(by='mean', descending=True)

def get_best_by_team(result_per_run_df, with_baselines=True):
    best_by_team = result_per_run_df.filter(pl.col('team') != 'baseline').unique(subset='team', keep='first', maintain_order=True)
    if not with_baselines:
        return best_by_team

    baseline_runs = result_per_run_df.filter(pl.col('team') == 'baseline')
    return pl.concat([best_by_team, baseline_runs]).sort('mean', descending=True).drop('run_id')

In [13]:
results_per_run25 = get_results_per_run(pan25_df)
results_per_run26 = get_results_per_run(pan26_df)

In [11]:
display(Markdown('# PAN\'25 Dataset:'))
with pl.Config(float_precision=3, tbl_rows=-1):
    display(get_best_by_team(results_per_run25))

# PAN'25 Dataset:

team,software,dataset,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""mdok""","""mdok""","""pan25-generative-ai-detection-…",0.997,0.988,0.986,0.990,0.994,0.991,0.006,0.017
"""plagiarism-detector-com-reg2""","""alternating-contingency""","""pan25-generative-ai-detection-…",0.999,0.986,0.983,0.981,0.992,0.988,0.001,0.024
"""turingteam""","""marinated-edge""","""pan25-generative-ai-detection-…",0.995,0.979,0.980,0.983,0.991,0.986,0.010,0.025
"""dsgt-pan-26""","""relaxed-copy""","""pan25-generative-ai-detection-…",0.998,0.977,0.972,0.979,0.990,0.983,0.005,0.038
"""suspiciously-coherent""","""lead-log""","""pan25-generative-ai-detection-…",0.998,0.980,0.970,0.979,0.967,0.979,0.098,0.001
"""styloch2026""","""instant-tartar""","""pan25-generative-ai-detection-…",0.988,0.967,0.965,0.975,0.989,0.977,0.003,0.048
"""sst""","""DeBERTa-V3-large-individual""","""pan25-generative-ai-detection-…",0.987,0.969,0.965,0.975,0.987,0.977,0.010,0.045
"""ak-ys""","""Qwen3-0.6B-fine-tuned (dockerf…","""pan25-generative-ai-detection-…",0.991,0.957,0.949,0.962,0.982,0.968,0.009,0.069
"""zhengqiaozeng""","""three""","""pan25-generative-ai-detection-…",0.978,0.931,0.929,0.946,0.976,0.952,0.006,0.099


In [12]:
display(Markdown('# PAN\'26 Dataset:'))
with pl.Config(float_precision=3, tbl_rows=-1):
    display(get_best_by_team(results_per_run26))

# PAN'26 Dataset:

team,software,dataset,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""suspiciously-coherent""","""lead-log""","""pan26-generative-ai-detection-…",0.989,0.965,0.965,0.976,0.980,0.975,0.053,0.029
"""sst""","""ModernBERT-large-BNN""","""pan26-generative-ai-detection-…",0.993,0.970,0.962,0.975,0.969,0.974,0.108,0.014
"""mdok""","""mdok""","""pan26-generative-ai-detection-…",0.995,0.951,0.946,0.963,0.984,0.968,0.002,0.072
"""plagiarism-detector-com-reg2""","""alternating-contingency""","""pan26-generative-ai-detection-…",0.996,0.941,0.935,0.899,0.957,0.946,0.000,0.072
"""dsgt-pan-26""","""relaxed-copy""","""pan26-generative-ai-detection-…",0.981,0.885,0.873,0.907,0.961,0.921,0.000,0.170
"""ydong""","""arid-winch""","""pan26-generative-ai-detection-…",0.930,0.915,0.886,0.928,0.896,0.911,0.420,0.014
"""turingteam""","""marinated-edge""","""pan26-generative-ai-detection-…",0.957,0.882,0.846,0.851,0.925,0.892,0.053,0.179
"""writerslogic-inc""","""large-rank""","""pan26-generative-ai-detection-…",0.891,0.891,0.853,0.902,0.899,0.887,0.312,0.092
"""a-a-detection""","""navy-iceberg""","""pan26-generative-ai-detection-…",0.852,0.848,0.826,0.885,0.880,0.858,0.375,0.106
